# Hands-On Machine Learning with PyTorch (Aurelion Geron)

## 1) Building Neural Networks with PyTorch

### 3. Nonsequential Models

#### Wide & Deep NN (Single Input)

In [1]:
import torch 

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()

X_train, X_test, y_train, y_test = train_test_split(housing.data, housing.target, random_state=42)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

mean = X_train.mean(dim=0, keepdims=True)
std = X_train.std(dim=0, keepdims=True)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std 

y_train = torch.FloatTensor(y_train).reshape(-1,1)
y_test = torch.FloatTensor(y_test).reshape(-1,1)

torch.manual_seed(42)

ImportError: /home/kobi/miniforge3/envs/zhang/lib/python3.14/site-packages/torch/../../.././libtorch_cuda.so: undefined symbol: ncclAlltoAll

In [2]:
import torch.nn as nn 

class WideAndDeep(nn.Module):

    def __init__(self, n_features):
        super().__init__()
        
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU())
        self.output_layer = nn.Linear(40 + n_features, 1)

    def forward(self, X):
        deep_output = self.deep_stack(X)
        wide_and_deep = torch.concat([X, deep_output], dim=1)

        return self.output_layer(wide_and_deep)

#### Wide & Deep NN (Multiple Output)

In [3]:
from torch.utils.data import TensorDataset, DataLoader 

class WideAndDeepV2(nn.Module):

    def __init__(self, n_features):
        super().__init__()

        self.deep_stack = nn.Sequential(
            nn.Linear(n_features-2, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
            nn.Linear(40, 30), nn.ReLU())

        self.output_layer = nn.Linear(30 + 5, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        
        return self.output_layer(wide_and_deep)


train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True, 
                             pin_memory=True, num_workers=2, prefetch_factor=2, 
                             persistent_workers=True)

test_data_wd = TensorDataset(X_test[:, :5], X_test[:, 2:], y_test)
test_loader_wd = DataLoader(test_data_wd, batch_size=32, shuffle=True,
                            pin_memory=True, num_workers=2, prefetch_factor=2,
                            persistent_workers=True)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

def train(model, optimizer, criterion, data_loader, n_epochs):
    model.train()

    for epoch in range(n_epochs):
        total_loss = 0

        for X_batch_wide, X_batch_deep, y_batch in data_loader:
            X_batch_wide = X_batch_wide.to(device)
            X_batch_deep = X_batch_deep.to(device)
            y_batch = y_batch.to(device)

            y_pred = model(X_batch_wide, X_batch_deep)

            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss/len(data_loader)
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss:.4f}.")

n_epochs = 20
n_features = X_train.shape[1]
lr = 0.01

model = WideAndDeepV2(n_features)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0)
mse = nn.MSELoss()


train(model, optimizer, mse, train_loader_wd, n_epochs)

def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []

    with torch.no_grad():
        for X_batch_wide, X_batch_deep, y_batch in data_loader:
            X_batch_wide = X_batch_wide.to(device)
            X_batch_deep = X_batch_deep.to(device)
            y_batch = y_batch.to(device)

            y_pred = model(X_batch_wide, X_batch_deep)

            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)

    return aggregate_fn(torch.stack(metrics))

eval_mse = evaluate(model, test_loader_wd, mse)

print(f"MSE on test set is {eval_mse}.")

# Alternatively, we can use the * unpack operator to unpack all the inputs


Epoch 1/20, Loss: 0.7969.
Epoch 2/20, Loss: 0.5043.
Epoch 3/20, Loss: 0.4715.
Epoch 4/20, Loss: 0.4499.
Epoch 5/20, Loss: 0.4358.
Epoch 6/20, Loss: 0.4247.
Epoch 7/20, Loss: 0.4077.
Epoch 8/20, Loss: 0.3962.
Epoch 9/20, Loss: 0.3821.
Epoch 10/20, Loss: 0.3753.
Epoch 11/20, Loss: 0.3684.
Epoch 12/20, Loss: 0.3580.
Epoch 13/20, Loss: 0.3575.
Epoch 14/20, Loss: 0.3546.
Epoch 15/20, Loss: 0.3512.
Epoch 16/20, Loss: 0.3498.
Epoch 17/20, Loss: 0.3443.
Epoch 18/20, Loss: 0.3406.
Epoch 19/20, Loss: 0.3359.
Epoch 20/20, Loss: 0.3365.
MSE on test set is 0.33976200222969055.


In [4]:
# We can avoid making mistakes when building the dataset (e.g., in order) by 
# introducing a custom dataset 

class WideAndDeepDataset(torch.utils.data.Dataset):
    def __init__(self, X_wide, X_deep, y):
        self.X_wide = X_wide 
        self.X_deep = X_deep 
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        input_dict = {"X_wide": self.X_wide[idx], "X_deep": self.X_deep[idx]}

        return input_dict, self.y[idx]

train_data_named = WideAndDeepDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_named = DataLoader(train_data_named, batch_size=32, shuffle=True)
# and so on ..., we just repeat the code from above ...

In [14]:
class WideAndDeepV3(nn.Module):

    def __init__(self, n_features):
        # same code as earlier first 
        super().__init__()
        
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features-2, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
            nn.Linear(40, 30), nn.ReLU())

        self.output_layer = nn.Linear(30+5, 1)
        self.aux_output_layer = nn.Linear(30, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep) 
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        main_output = self.output_layer(wide_and_deep)
        aux_output = self.aux_output_layer(deep_output)

        return main_output, aux_output # the model now predicts 2 outputs 


def train(model, optimizer, criterion, data_loader, n_epochs):
    model.train()

    for epoch in range(n_epochs):
        total_loss = 0

        for *X_batch, y_batch in data_loader:
            X_batch = [X.to(device) for X in X_batch]
            y_batch = y_batch.to(device)

            y_pred, y_pred_aux = model(*X_batch)
            main_loss = criterion(y_pred, y_batch)
            aux_loss = criterion(y_pred_aux, y_batch)

            loss = 0.8*main_loss + 0.2*aux_loss 
            total_loss += loss.item()
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss/len(data_loader)

        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss:.4f}.")

def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []

    with torch.no_grad():
        for *X_batch, y_batch in data_loader:
            X_batch = [X.to(device) for X in X_batch]
            y_batch = y_batch.to(device)

            y_pred, _ = model(*X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)

    return aggregate_fn(torch.stack(metrics))


model = WideAndDeepV3(n_features)
model = model.to(device)
train(model, optimizer, mse, train_loader_wd, n_epochs)

eval_mse = evaluate(model, test_loader_wd, mse)

print(f"Evaluation on test data with WideAndDeepV3 gives {eval_mse}.")

    

Epoch 1/20, Loss: 5.0151.
Epoch 2/20, Loss: 5.0153.
Epoch 3/20, Loss: 5.0154.
Epoch 4/20, Loss: 5.0163.
Epoch 5/20, Loss: 5.0155.
Epoch 6/20, Loss: 5.0155.
Epoch 7/20, Loss: 5.0162.
Epoch 8/20, Loss: 5.0146.
Epoch 9/20, Loss: 5.0162.
Epoch 10/20, Loss: 5.0156.
Epoch 11/20, Loss: 5.0158.
Epoch 12/20, Loss: 5.0152.
Epoch 13/20, Loss: 5.0150.
Epoch 14/20, Loss: 5.0153.
Epoch 15/20, Loss: 5.0158.
Epoch 16/20, Loss: 5.0152.
Epoch 17/20, Loss: 5.0159.
Epoch 18/20, Loss: 5.0148.
Epoch 19/20, Loss: 5.0165.
Epoch 20/20, Loss: 5.0142.
Evaluation on test data with WideAndDeepV3 gives 5.00345516204834.
